# RenderForge TTS — Audio Generator

Generate narration audio for RenderForge videos using **Qwen3-TTS**.

### Workflow
1. Upload `scripts.json` (generated by `npx tsx content/generate-scripts.ts`)
2. Choose voice style (Voice Design or Voice Clone)
3. Generate `.wav` files for each section
4. Download zip → place in `content/audio/` → run `audio-sync.ts`

### Requirements
- **GPU Runtime**: Go to Runtime → Change runtime type → T4 (free) or A100 (faster)
- **RAM**: ~8GB for 0.6B model, ~16GB for 1.7B model

> T4 (free tier) works with the 0.6B model. A100 recommended for 1.7B.

## Step 1: Install Dependencies

In [ ]:
!pip install -U qwen-tts soundfile numpy -q

# Optional: Flash Attention for faster generation (works on A100/T4)
!pip install -U flash-attn --no-build-isolation -q 2>/dev/null || echo "Flash Attention not available — falling back to default attention"

## Step 2: Upload scripts.json

Upload the `scripts.json` file generated by:
```bash
npx tsx content/generate-scripts.ts --limit 10
```

In [ ]:
from google.colab import files
import json

# Upload scripts.json
uploaded = files.upload()

scripts = json.loads(list(uploaded.values())[0].decode('utf-8'))
total_sections = sum(len(s['sections']) for s in scripts)

print(f"\nLoaded {len(scripts)} posts")
print(f"Total audio sections to generate: {total_sections}")
print(f"\nPosts:")
for s in scripts[:5]:
    print(f"  [{s['postId']}] {s['template']} — {s['title'][:50]}")
if len(scripts) > 5:
    print(f"  ... and {len(scripts) - 5} more")

## Step 3: Configure Voice

Choose your voice style. Options:
- **Voice Design**: Describe the voice you want in natural language
- **Voice Clone**: Upload a reference audio to clone a specific voice

For "Your Last Dollar" channel, we recommend a **confident, energetic male voice**.

In [ ]:
# ═══════════════════════════════════════════
#  VOICE CONFIGURATION
# ═══════════════════════════════════════════

# Choose mode: "design" or "clone"
VOICE_MODE = "design"

# ── Voice Design Settings ──
# Describe the voice you want in natural language
VOICE_INSTRUCT = (
    "A confident, energetic young male voice with a motivational tone. "
    "Clear pronunciation, medium-fast pace, slightly deep pitch. "
    "Sounds like a successful entrepreneur giving advice to friends. "
    "Natural and engaging, not robotic."
)

# ── Voice Clone Settings (only if VOICE_MODE = "clone") ──
# Upload a short reference audio (5-15 seconds of clear speech)
CLONE_REF_AUDIO = None  # Will be set in the clone cell below
CLONE_REF_TEXT = ""     # Transcription of the reference audio

# ── Model Settings ──
# Use 0.6B for free T4 GPU, 1.7B for A100
MODEL_SIZE = "0.6B"  # "0.6B" or "1.7B"
LANGUAGE = "English"

print(f"Voice mode: {VOICE_MODE}")
print(f"Model size: {MODEL_SIZE}")
print(f"Language: {LANGUAGE}")
if VOICE_MODE == "design":
    print(f"Voice description: {VOICE_INSTRUCT[:80]}...")

## Step 3b: (Optional) Upload Reference Audio for Voice Clone

Only run this cell if `VOICE_MODE = "clone"`. Upload a `.wav` file with 5-15 seconds of clear speech.

In [ ]:
# Only run this if VOICE_MODE = "clone"
if VOICE_MODE == "clone":
    print("Upload a reference audio file (.wav, 5-15 seconds of clear speech):")
    ref_upload = files.upload()
    ref_filename = list(ref_upload.keys())[0]
    CLONE_REF_AUDIO = ref_filename
    print(f"\nReference audio: {ref_filename}")
    print("\nNow enter the transcription of this audio:")
    CLONE_REF_TEXT = input("Transcription: ")
    print(f"Reference text: {CLONE_REF_TEXT}")
else:
    print("Skipped — using Voice Design mode")

## Step 4: Load Model

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

# Select model based on mode and size
if VOICE_MODE == "design":
    model_name = f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-VoiceDesign"
elif VOICE_MODE == "clone":
    model_name = f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base"
else:
    model_name = f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice"

print(f"Loading model: {model_name}")
print("This may take 2-5 minutes on first run (downloading weights)...")

# Try flash attention, fall back to eager
try:
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    print("Loaded with Flash Attention 2")
except Exception:
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0",
        dtype=torch.bfloat16,
    )
    print("Loaded with default attention")

print(f"Model loaded on {next(model.parameters()).device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 5: Generate Audio

This generates a `.wav` file for each section of each post. Progress is shown with ETA.

In [ ]:
import soundfile as sf
import numpy as np
import os
import time

os.makedirs("audio", exist_ok=True)

def generate_audio(text):
    """Generate audio using configured voice mode."""
    if VOICE_MODE == "design":
        wavs, sr = model.generate_voice_design(
            text=text,
            language=LANGUAGE,
            instruct=VOICE_INSTRUCT,
        )
    elif VOICE_MODE == "clone":
        wavs, sr = model.generate_voice_clone(
            text=text,
            language=LANGUAGE,
            ref_audio=CLONE_REF_AUDIO,
            ref_text=CLONE_REF_TEXT,
        )
    else:
        wavs, sr = model.generate_custom_voice(
            text=text,
            language=LANGUAGE,
            speaker="Ryan",
        )
    return wavs[0], sr

# Count total sections
total = sum(len(s['sections']) for s in scripts)
done = 0
failed = []
start_time = time.time()

print(f"Generating {total} audio files for {len(scripts)} posts...\n")
print("=" * 60)

for post in scripts:
    post_id = post['postId']
    post_dir = os.path.join("audio", post_id)
    os.makedirs(post_dir, exist_ok=True)

    print(f"\n[{post_id}] {post['template']} — {post['title'][:50]}")

    post_wavs = []
    sr = None

    for section in post['sections']:
        done += 1
        elapsed = time.time() - start_time
        per_item = elapsed / done if done > 0 else 0
        eta = per_item * (total - done)
        eta_str = f"{int(eta // 60)}m {int(eta % 60)}s" if eta > 60 else f"{int(eta)}s"

        output_path = os.path.join("audio", section['audioFile'])
        text = section['text']

        print(f"  [{done}/{total}] {section['key']}: \"{text[:60]}{'...' if len(text) > 60 else ''}\" (ETA: {eta_str})", end="")

        try:
            wav, current_sr = generate_audio(text)
            sf.write(output_path, wav, current_sr)
            duration = len(wav) / current_sr
            print(f" ✓ {duration:.1f}s")

            post_wavs.append(wav)
            if sr is None:
                sr = current_sr
        except Exception as e:
            print(f" ✗ {str(e)[:80]}")
            failed.append({'post': post_id, 'section': section['key'], 'error': str(e)})

    # Also save full narration (all sections concatenated)
    if post_wavs and sr:
        full_wav = np.concatenate(post_wavs)
        full_path = os.path.join(post_dir, "full.wav")
        sf.write(full_path, full_wav, sr)
        full_duration = len(full_wav) / sr
        print(f"  → Full narration: {full_duration:.1f}s")

total_time = time.time() - start_time
print("\n" + "=" * 60)
print(f"DONE! Generated {done - len(failed)}/{total} audio files in {int(total_time // 60)}m {int(total_time % 60)}s")
if failed:
    print(f"\nFailed ({len(failed)}):")
    for f in failed:
        print(f"  {f['post']}/{f['section']}: {f['error'][:80]}")

## Step 6: Preview Audio

Listen to a sample before downloading everything.

In [ ]:
from IPython.display import Audio, display
import os

# Preview the first post's full narration
first_post = scripts[0]['postId']
preview_path = f"audio/{first_post}/full.wav"

if os.path.exists(preview_path):
    print(f"Preview: {first_post} — {scripts[0]['title']}")
    display(Audio(preview_path))
else:
    # Try first section
    first_section = scripts[0]['sections'][0]['audioFile']
    alt_path = f"audio/{first_section}"
    if os.path.exists(alt_path):
        print(f"Preview: {first_post} intro")
        display(Audio(alt_path))
    else:
        print("No audio files found to preview.")

## Step 7: Download

Downloads a zip file. Extract it into your RenderForge `content/audio/` directory, then run:
```bash
npx tsx content/audio-sync.ts --limit 10
```

In [ ]:
import shutil
import os

# Show what was generated
total_size = 0
file_count = 0
for root, dirs, fnames in os.walk("audio"):
    for f in fnames:
        if f.endswith('.wav'):
            total_size += os.path.getsize(os.path.join(root, f))
            file_count += 1

print(f"Audio files: {file_count}")
print(f"Total size: {total_size / 1024 / 1024:.1f} MB")

# Create zip
print("\nCreating zip...")
shutil.make_archive("renderforge-audio", "zip", ".", "audio")
zip_size = os.path.getsize("renderforge-audio.zip") / 1024 / 1024
print(f"Zip created: renderforge-audio.zip ({zip_size:.1f} MB)")

# Download
print("\nStarting download...")
from google.colab import files
files.download("renderforge-audio.zip")

print("\n" + "=" * 60)
print("NEXT STEPS:")
print("1. Extract zip into your renderforge/content/ directory")
print("   (so files are at content/audio/day01-post1/intro.wav etc.)")
print("2. Run: npx tsx content/audio-sync.ts --limit 10")
print("3. Find synced videos in output/synced/story/")
print("=" * 60)

---

## Alternative Voice Styles

Change `VOICE_INSTRUCT` in Step 3 to try different voices:

| Channel | Voice Description |
|---------|------------------|
| **Your Last Dollar** | `"A confident, energetic young male voice with a motivational tone. Clear pronunciation, medium-fast pace, slightly deep pitch."` |
| **KnockKnockZoo** (kids) | `"A cheerful, playful young child voice. Bright and excited tone, slightly high pitch, fun and energetic. Like a happy 8-year-old telling jokes."` |
| **Stoicism/Motivation** | `"A calm, deep, wise male voice. Slow deliberate pace, authoritative but warm. Like a mentor sharing timeless wisdom."` |
| **Horror/Creepy** | `"A quiet, slightly whispery male voice. Slow pace with dramatic pauses. Unsettling and mysterious tone."` |
| **Tech/AI News** | `"A clear, professional, neutral male voice. Medium pace, informative tone. Like a tech journalist reporting news."` |
| **Dubai Luxury** | `"A smooth, sophisticated male voice with a slight British accent. Elegant and refined tone, medium-slow pace."` |